Correct warpcoefficient libraries such that templates can be generated with a color distribution matching the observed.

- specify a warp class
- read color distribution from output from warptemplate_colors, especially peak color estimate
- iterate through all warpcoefficients
- determine peak color
- determine correction spectrum to correct correction factor to have same color as sample mean
- record corrected warpfactors, together with original color as well as sample fit properties directly into warp collection.

Here we investigate whether we can find a useful relationship between g-R sample distribution, individual peak g-R color and E(B-V) color to be applied. 

In [ ]:
import os, re, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#from scipy.optimize import curve_fit
#from scipy.stats import exponnorm
from pathlib import Path
from datetime import datetime
import sncosmo
import extinction
from scipy.optimize import brentq
import pickle

In [ ]:
sys.path.append('/Users/jnordin/github/ampelFeb25')

In [ ]:
from warpTemplate import WarpfitTemplateLoader

In [ ]:
template_class_id = 1

In [ ]:
# NOw you have to decide which warp class to fit against:
warpclasses = [
    'SN Ib', 'SN Ia-91bg', 'SN II', 'SN Ia-91T',
    'SN Ib/c', 'SN Ia-pec', 'SN IIP', 'SN Iax', 'SN Ic',
    'SLSN-II']

In [ ]:
print('Fitting to templates of class', warpclasses[template_class_id])

In [ ]:
# Help methods to get mean color for model
def get_latest_model_result(model_name, infile="warptemplate_color_fits.csv"):
    file = Path(infile)

    if not file.exists():
        raise FileNotFoundError(f"{infile} does not exist")

    # Load data
    df = pd.read_csv(file)

    if df.empty:
        return None

    # Ensure timestamp is datetime
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # Filter by model
    df_model = df[df["model"] == model_name]

    if df_model.empty:
        return None

    # Get latest entry
    latest_row = df_model.sort_values("timestamp", ascending=False).iloc[0]

    return latest_row.to_dict()

In [ ]:
model_colors = get_latest_model_result(warpclasses[template_class_id])

In [ ]:
model_colors

In [ ]:
K, loc, scale = model_colors['K'], model_colors['loc'], model_colors['scale']
from scipy.stats import exponnorm
emg_dist = exponnorm(K, loc, scale)

In [ ]:
d2 = emg_dist.rvs(size=1000000, random_state=41)
_ = plt.hist(d2,bins=100)
plt.axvline(x=loc,color='red')
plt.axvline(x=loc+scale,color='green')

In [ ]:
emg_dist.rvs(size=1, random_state=rng.randint(0,100))

In [ ]:
import random
rng = random.Random(41)

In [ ]:
rng.randint(0,100)

In [ ]:
# Now we know which peak color the warped templates should be shifted 

In [ ]:
# Parameters for fit template retrieval
# How to define templates?
# - How many templates per sn basis? 
#      * if 'all' it will return one copy of each template, 
#.     * if int it will return that many, drawn according to the template probability, 
#      * if -int it will return that many copies drawn from a uniform probabilitiy
#      Note: draws made with replacement, so multiple copies can be returned if int is larger than the available number of templates (often 3)
template_selection = 'all'    # Use the same number per SN to keep balance 
# - How many sn basis?
#.     * if 'all', take one of each
#.     * if an int, draw these randomly (with replacement)
#.     Note: how many templates are returned is decided by the above parameter.
snbasis_selection = 'all'

warpdir = '/Users/jnordin/data/models/sncosmo/warpmod'

In [ ]:
warploader = WarpfitTemplateLoader(warpdir)

In [ ]:
templates = warploader.get_templates(
    fitclass=warpclasses[template_class_id],
    exclude_input = [],    # Here we do not wish to exclude anything 
    template_selection=template_selection,
    snbasis_selection=snbasis_selection,
    random_seed=42
)

In [ ]:
# Now we wish to loop through all sncosmo templates and:
# - measure the template peak phase color
# - calculate which extinction would need to be applied to this template to achieve the class color 
# - create a modified version of the template warp coefficients that lead to the desired color
# - store the new version 

In [ ]:
    def color_with_ebv(warped_model, ebv, rv, band1,band2):
        warped_model.set(hostebv=ebv, hostr_v=rv)
        return (
            warped_model.bandmag(band1, 'ab', 0)
            - warped_model.bandmag(band2, 'ab', 0)
        )


In [ ]:
ebvout = {}
colfits = []
for k, t in enumerate(templates):

    # This 
    mod = t['model']
    modid = mod.description
    source = mod.source


    # Help functions
    warped_model = sncosmo.Model(
        source=source,
        effects=[sncosmo.CCM89Dust()],
        effect_names=['host'],
        effect_frames=['rest']
    )
    # How to do with readshift? Are we talking about observed peak color, or restframe peak color?
#    warped_model.set(z=mod.get('z'))
    warped_model.set(z=0)

    # We first determine the native peak col
    natcol = color_with_ebv(warped_model, 0, 3.1, model_colors['color1'], model_colors['color2'])
    print(k, modid, natcol)

    def fit_function( ebv ):
        return color_with_ebv(warped_model, ebv, 3.1, model_colors['color1'], model_colors['color2'])-target_col
    
    # We now generate and loop through a number of simulated peak colors 
    for target_col in list(emg_dist.rvs(size=100)):
        # Determine which ebv is needed 
        # Look for the dust color ebv minimizing the difference
        try:
            ebv_solution = brentq(fit_function, -5.0, 5.0)

            # Store for analysis
            colfits.append(
                {
                    'k': k,
                    'sn': modid,
                    'z': mod.get('z'),
                    'natcol': natcol,
                    'targetcol': target_col,
                    'ebvfit': ebv_solution
                }
            )
                    
        except ValueError:
            # fallback if root not bracketed
            print('fit fail')
            ebv_solution = -99

In [ ]:
dfcol = pd.DataFrame.from_dict( colfits )

In [ ]:
plt.scatter( dfcol['targetcol'], dfcol['ebvfit'] )

In [ ]:
plt.scatter( dfcol['targetcol']-dfcol['natcol'], dfcol['ebvfit'] )
plt.grid(linestyle="--",alpha=0.5,zorder=1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = dfcol['targetcol'] - dfcol['natcol']
y = dfcol['ebvfit']

# Fit quadratic
coeffs = np.polyfit(x, y, 3)
poly = np.poly1d(coeffs)

# Generate smooth curve
x_fit = np.linspace(x.min(), x.max(), 300)
y_fit = poly(x_fit)

In [ ]:
plt.figure(figsize=(7,5))

hb = plt.hexbin(x, y, gridsize=60, cmap='viridis', bins='log')
plt.colorbar(hb, label='log(N)')

plt.plot(x_fit, y_fit, color='red', linewidth=2, label='Color fit')

plt.grid(linestyle="--", alpha=0.3)
plt.legend()

plt.xlabel(r"$(g-R)_{draw}$ - $(g-R)_{template}$")
plt.ylabel("Apply E(B-V)")
plt.tight_layout()
plt.show()

In [ ]:
model_colors

In [ ]:
model_colors['ebv_corr_func'] = list(coeffs)

In [ ]:
warploader._cache.keys()

In [ ]:
if template_class_id == 4:
    data = warploader._cache['SN Ibc']
else:
    data = warploader._cache[warpclasses[template_class_id]]

In [ ]:
newdata = []
for snwarpstuff in data:
    snwarpstuff['model_colors'] = model_colors
    newdata.append( snwarpstuff )


In [ ]:
len(data)

In [ ]:
len(newdata)

In [ ]:
key = re.sub(r"/", "", warpclasses[template_class_id])

In [ ]:
key

In [ ]:
newfilepath = os.path.join(
            warpdir,
            f"warpcoeffs_{key}_distinfo.pkl"
        )

In [ ]:
with open(newfilepath, "wb") as f:
    pickle.dump( newdata, f)